# 06 — Demo, project summary, and final README

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

The capstone. It produces three things, none of which contains a hand-typed result:

1. **`README.md`**: every number is read from the summary JSONs written by notebooks 1–5. A
   missing field becomes a visible `TODO`, never a guess. A claim ("X is better than Y",
   "uncertainty flags errors") appears only when the patient-bootstrap CI supports it.
2. **`results/final/project_summary.png`**: one figure that tells the whole story.
3. **A demo**: `predict_and_explain(image)` returns the prediction, MC-dropout confidence and
   uncertainty, Grad-CAM and IG overlays, and a **trust flag** whose thresholds come from the
   project's own test-set distributions. It's demonstrated here on real test images, and saved as
   a Gradio `app.py` for Hugging Face Spaces.

**Inputs to attach** (Add Input → Your Work → Notebooks). That's six:

| input | used for |
|---|---|
| notebook 0 | `manifest.csv` + `.mat` files (raw MRI for the demo) |
| notebook 1 | `best_model.pt`, `test_metrics.json`, `src/` → `data/`, `models/`, `utils/` |
| notebook 2 | `meta.json`, `test_explanations.npz`, `src/explain/` |
| notebook 3 | `audit_summary.json`, `audit_config.json`, `audit_results.csv`, audit figures |
| notebook 4 | `uncertainty_summary.json`, `uncertainty_results.csv`, `src/uncertainty/` |
| notebook 5 | `joint_summary.json`, `joint_results.csv`, joint figures |

CPU is enough. A GPU only makes the demo's Integrated Gradients faster.

## 1. Setup

`captum` is needed by `src/explain/integrated_gradients.py`. Gradio is installed **only** if you
switch on the live app in §9; the committed run doesn't need it.

In [ ]:
!pip install -q captum

In [ ]:
import hashlib
import importlib
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display
from skimage import measure

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "(CPU is fine)")

WORK = Path("/kaggle/working")
FINAL = WORK / "results" / "final"
(FINAL / "figures").mkdir(parents=True, exist_ok=True)

## 2. Locate all six inputs

Each artifact must exist **exactly once** under `/kaggle/input`, at any depth: Kaggle currently
mounts notebook outputs at `/kaggle/input/notebooks/<user>/<slug>/`. Zero copies stops the
notebook with a pointer to the missing notebook. Two copies stops it too, since that usually means
two versions of one notebook are attached.

In [ ]:
def find_unique(pattern, parent_names, what, how_to_fix):
    hits = [h for h in sorted(Path("/kaggle/input").rglob(pattern))
            if [p.name for p in h.parents[:len(parent_names)]] == parent_names]
    if len(hits) == 0:
        print("mounted under /kaggle/input:")
        for p in sorted(Path("/kaggle/input").rglob("*")):
            if p.is_dir() and len(p.relative_to("/kaggle/input").parts) <= 4:
                print("   ", p)
        raise FileNotFoundError(f"{what} not found under /kaggle/input. {how_to_fix}")
    if len(hits) > 1:
        raise RuntimeError(f"several copies of {what} attached, keep only one: {hits}")
    return hits[0]


CKPT_PATH = find_unique("best_model.pt", ["checkpoints", "results"], "notebook 1 checkpoint", "Attach notebook 1's output.")
TEST_METRICS = find_unique("test_metrics.json", ["results"], "notebook 1 test_metrics.json", "Attach notebook 1's output.")
EXPL_META = find_unique("meta.json", ["explanations", "results"], "notebook 2 meta.json", "Attach notebook 2's output.")
AUDIT_SUM = find_unique("audit_summary.json", ["audit", "results"], "notebook 3 audit_summary.json", "Attach notebook 3's output.")
UNC_SUM = find_unique("uncertainty_summary.json", ["uncertainty", "results"], "notebook 4 uncertainty_summary.json", "Attach notebook 4's output.")
JOINT_SUM = find_unique("joint_summary.json", ["joint", "results"], "notebook 5 joint_summary.json", "Attach notebook 5's output.")

NB = {1: CKPT_PATH.parents[2], 2: EXPL_META.parents[2], 3: AUDIT_SUM.parents[2], 4: UNC_SUM.parents[2], 5: JOINT_SUM.parents[2]}
for k, d in NB.items():
    print(f"notebook {k}: {d}")
for p in [NB[1] / "src/models/classifier.py", NB[2] / "src/explain/gradcam.py", NB[4] / "src/uncertainty/mc_dropout.py",
          NB[2] / "results/explanations/test_explanations.npz", NB[3] / "results/audit/audit_config.json",
          NB[3] / "results/audit/audit_results.csv", NB[4] / "results/uncertainty/uncertainty_results.csv",
          NB[5] / "results/joint/joint_results.csv"]:
    if not p.exists():
        raise FileNotFoundError(f"missing {p} -- re-commit the notebook that writes it")

### MD5 lineage across notebooks 1–5

The checkpoint's own MD5 is computed here from the file. Every downstream notebook recorded the
checkpoint and manifest MD5 it worked from. All of them must agree, otherwise the README would
quote numbers from different models or splits side by side. The manifest for the demo images is
picked by that same MD5.

In [ ]:
J = lambda p: json.loads(Path(p).read_text())
test_metrics, expl_meta = J(TEST_METRICS), J(EXPL_META)
audit_sum, audit_cfg = J(AUDIT_SUM), J(NB[3] / "results/audit/audit_config.json")
unc_sum, joint_sum = J(UNC_SUM), J(JOINT_SUM)

CKPT_MD5 = hashlib.md5(CKPT_PATH.read_bytes()).hexdigest()
ckpt_raw = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
lineage = {
    "nb1 checkpoint file": (CKPT_MD5, ckpt_raw["manifest_md5"]),
    "nb1 test_metrics.json": (CKPT_MD5, test_metrics.get("manifest_md5", ckpt_raw["manifest_md5"])),
    "nb2 meta.json": (expl_meta["checkpoint_md5"], expl_meta["manifest_md5"]),
    "nb3 audit_config.json": (audit_cfg["checkpoint_md5"], audit_cfg["manifest_md5"]),
    "nb4 uncertainty_summary.json": (unc_sum["lineage"]["checkpoint_md5"], unc_sum["lineage"]["manifest_md5"]),
    "nb5 joint_summary.json": (joint_sum["lineage"]["checkpoint_md5"], joint_sum["lineage"]["manifest_md5"]),
}
display(pd.DataFrame(lineage, index=["checkpoint_md5", "manifest_md5"]).T)
if len({v[0] for v in lineage.values()}) != 1 or len({v[1] for v in lineage.values()}) != 1:
    raise RuntimeError("LINEAGE MISMATCH across notebooks 1-5 -- the outputs describe different checkpoints/splits. "
                       "Re-run the stale notebook(s) on the same checkpoint before building the README.")
MANIFEST_MD5 = ckpt_raw["manifest_md5"]
print(f"lineage OK -- checkpoint {CKPT_MD5}, manifest {MANIFEST_MD5}")

MANIFEST_PATH = DATA_ROOT = None
for m in sorted(Path("/kaggle/input").rglob("manifest.csv")):
    if hashlib.md5(m.read_bytes()).hexdigest() != MANIFEST_MD5:
        continue
    first = pd.read_csv(m, nrows=1).file_path.iloc[0]
    for root in [m.parent / "data", m.parent.parent / "data"]:
        if (root / first).exists():
            MANIFEST_PATH, DATA_ROOT = m, root
            break
    if MANIFEST_PATH:
        break
if MANIFEST_PATH is None:
    raise FileNotFoundError(f"no manifest.csv with md5 {MANIFEST_MD5} (+ data/). Attach notebook 0's output.")
print("manifest:", MANIFEST_PATH)

## 3. Consolidate one `src/` tree

The demo needs code from **three** notebooks at once: the model and data code (notebook 1), the
explainers (notebook 2) and MC-dropout (notebook 4). Python imports a package `src` from the first
`sys.path` entry that has one, and never merges two folders of the same name. With three input
folders on the path, two would be silently hidden.

So the tree is built explicitly, one owner per subpackage:

| subpackage | copied from |
|---|---|
| `src/__init__.py`, `src/data/`, `src/models/`, `src/utils/` | notebook 1 |
| `src/explain/` | notebook 2 |
| `src/uncertainty/` | notebook 4 |
| `src/demo/` | created empty here, filled in §7 |

Every `/kaggle/input` entry is removed from `sys.path`, and so are any `src.*` modules already
imported (in case this cell is re-run). Then `/kaggle/working` is added: the **parent** of the
merged `src/`, because the code imports `src.data…` / `src.explain…`, and an entry pointing *at*
`src/` itself would break those imports. In effect, the merged tree is the only `src` Python can see.

A drift check follows. Notebooks 2 and 4 also carry copies of notebook 1's files (they copied
`src/` forward). If any of those copies differs from notebook 1's original, it's printed, since
that would mean notebooks ran on different code.

In [ ]:
SRC = WORK / "src"
if SRC.exists():
    shutil.rmtree(SRC)
SRC.mkdir()
PLAN = [(1, ["__init__.py", "data", "models", "utils"]), (2, ["explain"]), (4, ["uncertainty"])]
for nb, items in PLAN:
    for it in items:
        s = NB[nb] / "src" / it
        if not s.exists():
            raise FileNotFoundError(f"notebook {nb} output has no src/{it}")
        (shutil.copytree if s.is_dir() else shutil.copy2)(s, SRC / it)
(SRC / "__init__.py").touch()
(SRC / "demo").mkdir()                 # filled by the %%writefile in section 7
(SRC / "demo" / "__init__.py").touch()

sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

md5f = lambda p: hashlib.md5(p.read_bytes()).hexdigest()
drift = []
for f in sorted((NB[1] / "src").rglob("*.py")):
    rel = f.relative_to(NB[1] / "src")
    for nb in [2, 4]:
        other = NB[nb] / "src" / rel
        if other.exists() and md5f(other) != md5f(f):
            drift.append(f"src/{rel}: notebook {nb}'s copy differs from notebook 1's")
print("code drift between notebooks:", drift or "none")

print("\nmerged tree:")
for p in sorted(SRC.rglob("*.py")):
    print("   ", p.relative_to(WORK))

In [ ]:
NEEDED = ["src.data.parse_mat", "src.data.dataset", "src.data.transforms", "src.models.classifier", "src.utils.seed",
          "src.explain", "src.explain.gradcam", "src.explain.integrated_gradients",
          "src.uncertainty", "src.uncertainty.mc_dropout", "src.uncertainty.metrics"]
for name in NEEDED:
    mod = importlib.import_module(name)
    assert Path(mod.__file__).resolve().is_relative_to(SRC.resolve()), f"{name} imported from {mod.__file__}"
    print(f"  ok  {name:35s} <- {Path(mod.__file__).relative_to(WORK)}")
print(f"all {len(NEEDED)} modules from notebooks 1, 2 and 4 resolve inside the merged {SRC}")

## 4. Read every number the README will quote

`G(source, "a.b.c")` reads a field from a summary. If the field is missing, it returns `None`, the
README shows **`TODO(source:a.b.c)`** at that spot, and the path is added to a list printed at the
end. Nothing is filled in by hand.

The notebooks store statistics in two shapes: notebook 3 uses `{mean, ci95_patient_bootstrap}`,
notebooks 4–5 use `{value, ci95_patient_bootstrap}`. `est()` reads either. `claim()` turns an
estimate into words **only if its CI excludes the null value**; otherwise it says the difference
isn't distinguishable from patient sampling noise.

In [ ]:
SOURCES = {"nb1": test_metrics, "nb2": expl_meta, "nb3": audit_sum, "nb3cfg": audit_cfg, "nb4": unc_sum, "nb5": joint_sum}
TODOS = []


def G(src, path):
    node = SOURCES[src]
    for part in path.split("|"):         # '|' separates keys, since keys contain '.', '+', '-'
        if not isinstance(node, dict) or part not in node:
            TODOS.append(f"{src}:{path}")
            return None
        node = node[part]
    return node


def est(entry):
    if not isinstance(entry, dict):
        return None, None
    v = entry.get("value", entry.get("mean"))
    ci = entry.get("ci95_patient_bootstrap")
    return v, ci


def f3(v, nd=3, signed=False):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "n/a"
    return f"{v:+.{nd}f}" if signed else f"{v:.{nd}f}"


def S(src, path, nd=3, signed=False, pct=False):
    # "value [lo, hi]" for a stat entry, or a TODO marker
    e = G(src, path)
    if e is None:
        return f"`TODO({src}:{path})`"
    v, ci = est(e) if isinstance(e, dict) else (e, None)
    k = 100 if pct else 1
    out = f3(v * k if v is not None else None, nd, signed) + ("%" if pct else "")
    if ci:
        out += f" [{f3(ci[0] * k, nd, signed)}, {f3(ci[1] * k, nd, signed)}]"
    return out


def supported(entry, null=0.0):
    _, ci = est(entry)
    return bool(ci) and np.isfinite(ci).all() and (ci[0] > null or ci[1] < null)


def claim(src, path, pos, neg, null_text, null=0.0):
    e = G(src, path)
    if e is None:
        return f"`TODO({src}:{path})`"
    v, _ = est(e)
    if not supported(e, null):
        return null_text
    return pos if v > null else neg


METHODS = ["gradcam", "integrated_gradients"]
LBL = {"gradcam": "Grad-CAM", "integrated_gradients": "Integrated Gradients", "random": "random-map control"}
CLASS_NAMES = expl_meta["class_names"]
SPECS = sorted(k[len("spearman_"):] for k in (G("nb3", "per_method|gradcam") or {}) if k.startswith("spearman_"))
print("stability transforms found in notebook 3's summary:", SPECS)

## 5. Project-summary figure

One figure, eight panels, read left to right as the argument of the project:

- **(a)** baseline confusion matrix (notebook 1)
- **(b–d)** the three audit axes, Grad-CAM vs IG vs the random control, with patient-bootstrap CIs
  (notebook 3)
- **(e)** calibration, deterministic vs MC (notebook 4)
- **(f)** how well each uncertainty measure detects errors, as AUROC (notebook 4)
- **(g–h)** the confidence × faithfulness quadrant (notebook 5)

All bars and error bars come from the summaries. Panels (e), (g) and (h) are drawn from the saved
per-image tables.

In [ ]:
from src.uncertainty import metrics as U

COL = {"gradcam": "#d95f02", "integrated_gradients": "#1b9e77", "random": "#9a9a9a"}
unc_res = pd.read_csv(NB[4] / "results/uncertainty/uncertainty_results.csv")
joint = pd.read_csv(NB[5] / "results/joint/joint_results.csv")


def bars(ax, metrics, methods, title, ylabel, null=None):
    x = np.arange(len(metrics))
    width = 0.8 / len(methods)
    for j, m in enumerate(methods):
        vals, lo, hi = [], [], []
        for met in metrics:
            v, ci = est(G("nb3", f"per_method|{m}|{met}"))
            vals.append(np.nan if v is None else v)
            lo.append(0 if not ci else v - ci[0]); hi.append(0 if not ci else ci[1] - v)
        ax.bar(x + (j - (len(methods) - 1) / 2) * width, vals, width * 0.92, yerr=[lo, hi], capsize=2,
               color=COL[m], alpha=0.85, label=LBL[m])
    ax.set_xticks(x, [mt.replace("_", "\n") for mt in metrics], fontsize=7)
    if null is not None:
        ax.axhline(null, color="k", lw=0.7, ls="--")
    ax.set_title(title, fontsize=9); ax.set_ylabel(ylabel, fontsize=8)


fig, axes = plt.subplots(2, 4, figsize=(22, 10))
cm = np.array(test_metrics["confusion_matrix"], float)
cmn = cm / cm.sum(1, keepdims=True)
ax = axes[0, 0]
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{int(cm[i, j])}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9, color="white" if cmn[i, j] > 0.6 else "black")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=8); ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"(a) baseline test confusion matrix\nacc {f3(test_metrics.get('accuracy'))}, balanced acc {f3(test_metrics.get('balanced_accuracy'))}", fontsize=9)

bars(axes[0, 1], ["deletion_auc", "insertion_auc"], METHODS + ["random"], "(b) faithfulness\n(deletion: lower better; insertion: higher better)", "AUC")
bars(axes[0, 2], [f"spearman_{s}" for s in SPECS] + [f"ssim_{s}" for s in SPECS], METHODS, "(c) stability (higher = more stable)\nSSIM inflated for sparse maps", "similarity")
bars(axes[0, 3], ["energy_ratio", "pointing_hit", "iou", "dice"], METHODS + ["random"], "(d) localization vs tumor mask\n(energy ratio: 1 = chance)", "score", null=1.0)
axes[0, 1].legend(fontsize=7, frameon=False)

ax = axes[1, 0]
for name, cols, pred_col, colr in [("deterministic", [f"p_det_{c}" for c in CLASS_NAMES], "det_pred", "#7570b3"),
                                   ("MC-averaged", [f"p_mc_{c}" for c in CLASS_NAMES], "mc_pred", "#e7298a")]:
    probs = unc_res[cols].to_numpy()
    rb = [b for b in U.reliability_bins(probs.max(1), unc_res[pred_col].to_numpy() == unc_res.true.to_numpy()) if b["count"]]
    ax.plot([b["confidence"] for b in rb], [b["accuracy"] for b in rb], "o-", color=colr, ms=4,
            label=f"{name}: ECE {S('nb4', 'calibration|' + ('deterministic_ece' if name == 'deterministic' else 'mc_ece'))}")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlim(0.3, 1.01); ax.set_ylim(0.3, 1.01)
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7, frameon=False)
ax.set_title("(e) calibration (reliability, 15 bins)", fontsize=9)

ax = axes[1, 1]
auc_names = ["predictive_entropy", "mutual_information", "expected_entropy", "1-det_confidence (baseline)"]
vals, err = [], [[], []]
for n_ in auc_names:
    v, ci = est(G("nb4", f"error_detection_auroc|{n_}"))
    vals.append(np.nan if v is None else v)
    err[0].append(0 if not ci else v - ci[0]); err[1].append(0 if not ci else ci[1] - v)
ax.barh(range(len(auc_names)), vals, xerr=err, color=["#e7298a"] * 3 + ["#7570b3"], alpha=0.8, capsize=3)
ax.axvline(0.5, color="k", lw=0.8, ls="--")
ax.set_yticks(range(len(auc_names)), auc_names, fontsize=8); ax.set_xlim(0, 1)
ax.set_title("(f) does uncertainty detect errors?\nAUROC, 0.5 = useless", fontsize=9)

for ax, m, tag in [(axes[1, 2], "gradcam", "g"), (axes[1, 3], "integrated_gradients", "h")]:
    sub = joint[joint.method == m]
    ok = sub.faith_score.notna()
    x = np.clip(1 - sub.mc_conf_explained, 1e-6, 1)
    ax.scatter(x[ok & sub.correct], sub.faith_score[ok & sub.correct], s=8, alpha=0.45, color="#555", label="correct")
    ax.scatter(x[ok & ~sub.correct], sub.faith_score[ok & ~sub.correct], s=40, marker="x", color="crimson", label="wrong")
    ax.axhline(0, color="#999", lw=0.8, ls=":")
    ax.axhline(sub.faith_score[ok].median(), color="k", lw=0.7, ls="--")
    ax.axvline((1 - sub.mc_conf_explained[ok]).median(), color="k", lw=0.7, ls="--")
    ax.set_xscale("log"); ax.invert_xaxis()
    r = G("nb5", f"correlations|{m}|mc_conf_explained~faith_score")
    ax.set_title(f"({tag}) {LBL[m]}: confidence vs faithfulness\nSpearman {S('nb5', f'correlations|{m}|mc_conf_explained~faith_score')}", fontsize=9)
    ax.set_xlabel("1 - MC confidence (log; left = confident)", fontsize=8); ax.set_ylabel("faith_score (0 = random)", fontsize=8)
    ax.legend(fontsize=7, frameon=False)
fig.suptitle("Quantitatively audited XAI for brain tumor MRI: is a convincing heatmap a trustworthy one?", fontsize=14, y=1.0)
plt.tight_layout()
fig.savefig(FINAL / "project_summary.png", dpi=150, bbox_inches="tight")
plt.show()

for nb, names in [(3, ["audit_deletion_insertion_curves.png", "audit_stability_distributions.png", "audit_localization_distributions.png"]),
                  (4, ["uncertainty_reliability.png", "uncertainty_error_detection.png"]),
                  (5, ["joint_quadrant_scatter.png", "joint_correlation_heatmap.png", "joint_exemplars.png"])]:
    for n_ in names:
        p = NB[nb] / "results/figures" / n_
        if p.exists():
            shutil.copy2(p, FINAL / "figures" / n_)
        else:
            print(f"figure not found (README will skip it): notebook {nb} {n_}")
print("copied figures:", sorted(p.name for p in (FINAL / "figures").iterdir()))

## 6. Trust-flag thresholds, from the project's own distributions

At inference there's **no tumor mask**, so the demo can't measure localization directly. Its
three warning signs must be computable from the image alone, and each threshold is read off the
saved test-set tables:

| signal | threshold | how it was chosen |
|---|---|---|
| low confidence | `mc_conf_explained` < *t₁* | 5th percentile among **correct** test predictions (95% of correct predictions are more confident) |
| high uncertainty | `predictive_entropy` > *t₂* | 95th percentile among **correct** test predictions |
| diffuse explanation (localization proxy) | Grad-CAM area ≥ 0.5 > *t₃* | 95th percentile of `heat_area_above_thr` among test images whose Grad-CAM peak **hit** the tumor (well-localized maps rarely spread wider than this) |

The flag is raised if **any** signal fires. "No flag" doesn't mean "trustworthy"; it means none of
the project's measured warning signs fired.

**This is checked on the same test set it was tuned on, so it's optimistic.** There's no held-out
set left to tune on, and the thresholds aren't tuned for any target metric, only read from
distributions. The back-test below says how many errors the flag would have caught, and whether
flagged images really have worse explanations. The diffuseness proxy gets its own honesty check:
its AUROC for predicting a pointing miss. If that's near 0.5, the proxy is weak, and both the
README and the app say so.

In [ ]:
gc = joint[joint.method == "gradcam"].sort_values("row").reset_index(drop=True)
correct = gc.correct.to_numpy(bool)
LOC_THR = audit_cfg["audit"]["localization_threshold"]
hit = gc.pointing_hit == 1
TH = {
    "min_confidence": float(np.quantile(gc.mc_conf_explained[correct], 0.05)),
    "max_entropy": float(np.quantile(gc.predictive_entropy[correct], 0.95)),
    "max_gradcam_area": float(np.quantile(gc.heat_area_above_thr[hit], 0.95)) if hit.any() else None,
    "gradcam_area_threshold": LOC_THR,
    "chosen_from": "notebook 5 joint_results.csv (Grad-CAM rows, test set): conf/entropy = 5th/95th pct of correct predictions; "
                   "area = 95th pct among pointing-game hits",
    "n_forward_passes": int(unc_sum["settings"]["n_forward_passes"]),
    "ig_steps": int(expl_meta["integrated_gradients"]["n_steps"]),
    "target_layer": expl_meta["gradcam"]["target_layer"],
    "class_names": CLASS_NAMES,
    "checkpoint_md5": CKPT_MD5,
}
print(json.dumps({k: v for k, v in TH.items() if k != "chosen_from"}, indent=2))

flag_conf = gc.mc_conf_explained < TH["min_confidence"]
flag_ent = gc.predictive_entropy > TH["max_entropy"]
flag_area = gc.heat_area_above_thr > TH["max_gradcam_area"] if TH["max_gradcam_area"] is not None else pd.Series(False, index=gc.index)
flag = (flag_conf | flag_ent | flag_area).to_numpy()

BOOT = U.patient_bootstrap_indices(gc.patient_id.to_numpy(), n_boot=int(unc_sum["settings"].get("n_bootstrap", 1000)), seed=int(unc_sum["settings"]["seed"]))
def bt(name, fn, *arrs):
    arrs = tuple(np.asarray(a, float) for a in arrs)
    v = float(fn(*arrs)); ci = U.bootstrap_ci(fn, arrs, BOOT)
    TH.setdefault("backtest", {})[name] = {"value": v, "ci95_patient_bootstrap": ci}
    return f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"

wrong = (~correct).astype(float)
print(f"flagged {flag.mean():.1%} of test images (conf {flag_conf.mean():.1%}, entropy {flag_ent.mean():.1%}, diffuse map {flag_area.mean():.1%})")
print("share of errors flagged      :", bt("error_recall", lambda f, w: f[w > 0].mean() if (w > 0).any() else np.nan, flag, wrong))
print("accuracy when flagged        :", bt("accuracy_flagged", lambda f, c: c[f > 0].mean() if (f > 0).any() else np.nan, flag, correct))
print("accuracy when not flagged    :", bt("accuracy_unflagged", lambda f, c: c[f == 0].mean() if (f == 0).any() else np.nan, flag, correct))
fs = gc.faith_score.to_numpy(float)
print("faith_score flagged - not    :", bt("faith_score_flagged_minus_unflagged",
      lambda f, s: np.nanmean(s[f > 0]) - np.nanmean(s[f == 0]) if (f > 0).any() and (f == 0).any() else np.nan, flag, fs))
print("diffuseness proxy -> pointing miss, AUROC:", bt("area_proxy_auroc_pointing_miss",
      lambda a, h: U.auroc(a[np.isfinite(h)], h[np.isfinite(h)] == 0), gc.heat_area_above_thr, gc.pointing_hit))

(FINAL / "trust_thresholds.json").write_text(json.dumps(TH, indent=2, default=float), encoding="utf-8")

## 7. The demo: `predict_and_explain(image)`

The inference logic lives in `src/demo/inference.py`, so the notebook and the Gradio app run
**exactly the same code**. `TumorExplainer` loads the checkpoint once (`load_checkpoint`, no
ImageNet download). Each call:

1. **Preprocesses** like `BrainTumorDataset`: a 2-D slice (raw int16 from a `.mat`, or a greyscale or
   RGB PNG) → per-image min-max → uint8 → 3 channels → the eval transform (resize 224, ImageNet
   normalise).
2. **Deterministic pass** (dropout off) → the predicted class, which is the class both explainers
   explain. This is the same convention notebooks 2–5 used.
3. **MC-dropout**: the image is stacked `T = 30` times in **one batch** with only dropout active.
   Each copy gets an independent dropout mask, and BN is in eval, so there's no batch interaction.
   That's equivalent to 30 separate passes, at a single forward pass's cost. The RNG is seeded per
   call, so the same image always gives the same numbers. Outputs: the MC mean confidence of the
   explained class, predictive entropy, mutual information.
4. **Grad-CAM and IG** with notebook 2's exact settings (`target_layer`, `n_steps`).
5. **Trust flag** from §6's thresholds, with the reasons spelled out.

In [ ]:
%%writefile /kaggle/working/src/demo/inference.py
import numpy as np
import torch
from matplotlib import colormaps
from torchvision import tv_tensors

from src.data.dataset import minmax_uint8
from src.data.transforms import get_eval_transforms, IMAGENET_MEAN, IMAGENET_STD
from src.explain import gradcam, integrated_gradients
from src.models.classifier import load_checkpoint
from src.uncertainty.mc_dropout import assert_mc_mode, enable_mc_dropout
from src.uncertainty.metrics import mc_decomposition

DISCLAIMER = "Research demo, not a medical device. Do not use for diagnosis."


def to_gray2d(image):
    a = np.asarray(image)
    if a.ndim == 3:                     # RGB(A) upload -> grey; the model only ever saw grey slices
        a = a[..., :3].astype(np.float32).mean(axis=2)
    if a.ndim != 2:
        raise ValueError(f"expected a 2-D slice or an RGB image, got shape {a.shape}")
    return a


def overlay(gray01, heat, alpha=0.45):
    rgb = np.repeat(gray01[..., None], 3, axis=2)
    hm = colormaps["jet"](np.clip(heat, 0, 1))[..., :3]
    return (np.clip((1 - alpha) * rgb + alpha * hm, 0, 1) * 255).astype(np.uint8)


class TumorExplainer:
    def __init__(self, ckpt_path, thresholds, device=None, seed=42):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model, ckpt = load_checkpoint(ckpt_path, self.device)
        self.class_names = ckpt["class_names"]
        self.image_size = ckpt["config"]["image_size"]
        self.tf = get_eval_transforms(self.image_size)
        self.th = thresholds
        self.T = int(thresholds["n_forward_passes"])
        self.ig_steps = int(thresholds["ig_steps"])
        self.target_layer = thresholds["target_layer"]
        self.seed = seed

    def preprocess(self, image):
        u8 = minmax_uint8(to_gray2d(image))
        x = self.tf(tv_tensors.Image(torch.from_numpy(u8).unsqueeze(0).repeat(3, 1, 1))).as_subclass(torch.Tensor)
        gray01 = (x[0] * IMAGENET_STD[0] + IMAGENET_MEAN[0]).clamp(0, 1).numpy()
        return x.float(), gray01

    @torch.no_grad()
    def _mc(self, x):
        enable_mc_dropout(self.model)
        assert_mc_mode(self.model)
        torch.manual_seed(self.seed)
        # one batch of T copies: each copy draws its own dropout mask; BN is in eval so copies don't interact
        p = self.model(x[None].to(self.device).repeat(self.T, 1, 1, 1)).float().softmax(1).cpu().numpy()
        self.model.eval()
        return p[:, None, :]            # (T, N=1, C)

    def __call__(self, image):
        x, gray01 = self.preprocess(image)
        self.model.eval()
        with torch.no_grad():
            det = self.model(x[None].to(self.device)).float().softmax(1)[0].cpu().numpy()
        target = int(det.argmax())
        samples = self._mc(x)
        d = mc_decomposition(samples)
        mc_probs = d["mean_probs"][0]
        gc = gradcam.explain(self.model, x, target=target, target_layer=self.target_layer)
        ig = integrated_gradients.explain(self.model, x, target=target, n_steps=self.ig_steps)

        conf = float(mc_probs[target])
        ent = float(d["predictive_entropy"][0])
        area = float((gc >= self.th["gradcam_area_threshold"]).mean())
        reasons = []
        if conf < self.th["min_confidence"]:
            reasons.append(f"MC confidence {conf:.3f} < {self.th['min_confidence']:.3f}")
        if ent > self.th["max_entropy"]:
            reasons.append(f"predictive entropy {ent:.3f} > {self.th['max_entropy']:.3f}")
        if self.th.get("max_gradcam_area") is not None and area > self.th["max_gradcam_area"]:
            reasons.append(f"diffuse Grad-CAM ({area:.1%} of image > {self.th['max_gradcam_area']:.1%})")
        return {
            "predicted_class": self.class_names[target],
            "class_index": target,
            "mc_confidence": conf,
            "mc_probs": {c: float(p) for c, p in zip(self.class_names, mc_probs)},
            "det_probs": {c: float(p) for c, p in zip(self.class_names, det)},
            "mc_predicted_class": self.class_names[int(mc_probs.argmax())],
            "predictive_entropy": ent,
            "mutual_information": float(d["mutual_information"][0]),
            "gradcam_area": area,
            "trust_flag": "LOW TRUST - review" if reasons else "no flag raised",
            "flag_reasons": reasons,
            "gradcam": gc, "ig": ig, "image": gray01,
            "gradcam_overlay": overlay(gray01, gc), "ig_overlay": overlay(gray01, ig),
        }

In [ ]:
import src.demo.inference as inference
importlib.reload(inference)
from src.data.parse_mat import read_mat

EXPLAINER = inference.TumorExplainer(CKPT_PATH, TH, device=DEVICE, seed=int(unc_sum["settings"]["seed"]))


def predict_and_explain(image):
    # image: 2-D MRI slice (raw .mat int16, or greyscale/RGB array); returns the dict described above
    return EXPLAINER(image)


print("model loaded once on", EXPLAINER.device, "| classes:", EXPLAINER.class_names, "| T =", EXPLAINER.T, "| IG steps =", EXPLAINER.ig_steps)

### Demonstration on real test images

Three images, chosen deterministically from notebook 5's joined table:
1. **confident and correct, with an above-median faithful Grad-CAM**: the most confident such image
2. **wrong**: the most confident error (the danger case, if one exists)
3. **most uncertain**: the highest predictive entropy

Each goes into `predict_and_explain` **from its raw `.mat` slice**, the way a new scan would.
Before the panels are drawn, two consistency checks run:
- the demo's prediction equals notebook 4's deterministic prediction
- the demo's Grad-CAM matches notebook 2's saved map (Spearman ≥ 0.99)

Together they show the demo really runs the audited pipeline and not a lookalike. The ground-truth
contour is drawn **for the reader only**; the model never sees it.

In [ ]:
from scipy.stats import spearmanr

z = np.load(NB[2] / "results/explanations/test_explanations.npz")
picks = []
ok_f = gc.faith_score.notna()
c1 = gc[gc.correct & ok_f & (gc.faith_score >= gc.faith_score[ok_f].median())]
if len(c1):
    picks.append(("confident + correct + faithful", int(c1.sort_values("mc_conf_explained", ascending=False).row.iloc[0])))
w_ = gc[~gc.correct]
if len(w_):
    picks.append(("wrong (most confident error)", int(w_.sort_values("mc_conf_explained", ascending=False).row.iloc[0])))
u_ = gc[~gc.row.isin([p[1] for p in picks])]
picks.append(("most uncertain", int(u_.sort_values("predictive_entropy", ascending=False).row.iloc[0])))
print(picks)

outs = []
for label, row in picks:
    r = gc[gc.row == row].iloc[0]
    o = predict_and_explain(read_mat(DATA_ROOT / r.file_path).image)
    rho = spearmanr(o["gradcam"].ravel(), z["gradcam"][row].astype(np.float32).ravel())[0]
    det_nb4 = CLASS_NAMES[int(unc_res.det_pred[unc_res.file_path == r.file_path].iloc[0])]
    print(f"{label:32s} demo pred {o['predicted_class']:10s} (nb4 det_pred {det_nb4:10s}) | Grad-CAM vs saved map Spearman {rho:.4f}")
    assert o["predicted_class"] == det_nb4, "demo prediction differs from the pipeline's"
    assert not np.isfinite(rho) or rho >= 0.99, "demo Grad-CAM differs from notebook 2's saved map"
    outs.append((label, r, o))

fig, axes = plt.subplots(len(outs), 4, figsize=(17, 4.3 * len(outs)))
axes = np.atleast_2d(axes)
for i, (label, r, o) in enumerate(outs):
    mask = z["mask"][int(r.row)]
    axes[i, 0].imshow(o["image"], cmap="gray")
    for c in measure.find_contours(mask.astype(float), 0.5):
        axes[i, 0].plot(c[:, 1], c[:, 0], color="lime", lw=1)
    axes[i, 0].set_title(f"[{label}]\ntrue: {CLASS_NAMES[int(r.true)]} (green = GT mask, not used by the model)", fontsize=9)
    axes[i, 1].imshow(o["gradcam_overlay"]); axes[i, 1].set_title("Grad-CAM (layer4, 7x7 upsampled)", fontsize=9)
    axes[i, 2].imshow(o["ig_overlay"]); axes[i, 2].set_title(f"Integrated Gradients ({EXPLAINER.ig_steps} steps)", fontsize=9)
    txt = (f"prediction: {o['predicted_class']}\nMC confidence: {o['mc_confidence']:.4f}\n"
           f"predictive entropy: {o['predictive_entropy']:.4f}\nmutual information: {o['mutual_information']:.5f}\n"
           f"Grad-CAM area >= {LOC_THR}: {o['gradcam_area']:.1%}\n\n"
           f"TRUST FLAG: {o['trust_flag']}\n" + "\n".join("  - " + s for s in o["flag_reasons"]) +
           f"\n\n(audit, not available at inference:\n faith_score {f3(r.faith_score)}, energy ratio {f3(r.energy_ratio)})")
    axes[i, 3].text(0.02, 0.98, txt, va="top", ha="left", fontsize=9, family="monospace",
                    color="crimson" if o["flag_reasons"] else "black", transform=axes[i, 3].transAxes)
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
fig.savefig(FINAL / "demo_examples.png", dpi=130, bbox_inches="tight")
plt.show()

## 8. Gradio app for Hugging Face Spaces

A live Gradio server doesn't survive a committed Kaggle run, so the app is **written to disk**:
- `app.py` wraps the same `TumorExplainer`
- `requirements.txt`
- `DEPLOY.md` with step-by-step Spaces instructions
- a copy of the checkpoint and `trust_thresholds.json`

Together with `src/`, the output folder is then a complete, deployable Space.

To try it live inside Kaggle, set `RUN_GRADIO = True` in §9. It's `False` by default, so **Run All
and the commit never block on a server**.

In [ ]:
APP = r'''
# Gradio demo: brain tumor MRI classification with audited explanations.
# Research demo only -- not a medical device.
import json
import os
from pathlib import Path

import gradio as gr

from src.data.parse_mat import read_mat
from src.demo.inference import DISCLAIMER, TumorExplainer

HERE = Path(__file__).parent
CKPT = os.environ.get("CKPT_PATH", str(HERE / "best_model.pt"))
TH = json.loads((HERE / "trust_thresholds.json").read_text(encoding="utf-8"))
if os.environ.get("IG_STEPS"):                 # CPU Spaces: fewer IG steps -> faster, slightly noisier maps
    TH["ig_steps"] = int(os.environ["IG_STEPS"])
EXPLAINER = TumorExplainer(CKPT, TH)


def run(image, mat_file):
    if mat_file:
        image = read_mat(mat_file).image
    if image is None:
        raise gr.Error("Upload an MRI slice image or a Figshare .mat file.")
    o = EXPLAINER(image)
    flag = "### " + ("LOW TRUST - send for review" if o["flag_reasons"] else "No warning sign fired")
    details = "\n".join(f"- {r}" for r in o["flag_reasons"]) or "- confidence, entropy and explanation spread all within the test-set range of correct predictions"
    text = (f"{flag}\n\n**Prediction:** {o['predicted_class']} (MC-dropout confidence {o['mc_confidence']:.3f})\n\n"
            f"**Predictive entropy:** {o['predictive_entropy']:.4f} nats | **Mutual information:** {o['mutual_information']:.5f}\n\n"
            f"**Why flagged:**\n{details}\n\n"
            "_Thresholds come from the project's test-set distributions (see trust_thresholds.json). "
            "'No warning sign' is not a guarantee of correctness._\n\n" + DISCLAIMER)
    return o["mc_probs"], o["gradcam_overlay"], o["ig_overlay"], text


demo = gr.Interface(
    fn=run,
    inputs=[gr.Image(type="numpy", label="MRI slice (PNG/JPG, axial T1-CE)"),
            gr.File(type="filepath", file_types=[".mat"], label="...or a Figshare .mat file")],
    outputs=[gr.Label(num_top_classes=3, label="MC-dropout class probabilities"),
             gr.Image(label="Grad-CAM"), gr.Image(label="Integrated Gradients"), gr.Markdown()],
    title="Audited XAI for brain tumor MRI",
    description="ResNet18 (meningioma / glioma / pituitary) with Grad-CAM, Integrated Gradients, "
                "MC-dropout uncertainty and a trust flag. " + DISCLAIMER,
)

if __name__ == "__main__":
    demo.launch()
'''.lstrip()
(WORK / "app.py").write_text(APP, encoding="utf-8")

(WORK / "requirements.txt").write_text(
    "# CPU is enough for the demo; pin versions after a successful Space build if you want reproducibility\n"
    "torch\ntorchvision\ncaptum\nnumpy\nscipy\nscikit-learn\nscikit-image\nh5py\nmatplotlib\npandas\ngradio\n",
    encoding="utf-8",
)

shutil.copy2(CKPT_PATH, WORK / "best_model.pt")
shutil.copy2(FINAL / "trust_thresholds.json", WORK / "trust_thresholds.json")

(WORK / "DEPLOY.md").write_text(f'''# Deploying the demo to Hugging Face Spaces

1. Create a new Space -> SDK **Gradio** -> hardware **CPU basic** (free).
2. Upload from this notebook's Output (or your repo):
   - `app.py`, `requirements.txt`, `trust_thresholds.json`
   - `best_model.pt` ({CKPT_PATH.stat().st_size / 1e6:.0f} MB; files over 10 MB need Git LFS if you push with git; the web uploader handles it)
   - the whole `src/` folder (merged tree: data, models, utils, explain, uncertainty, demo)
3. Optional Space variables: `IG_STEPS=20` makes Integrated Gradients about 2.5x faster on CPU (noisier maps);
   `CKPT_PATH` if the checkpoint lives elsewhere.
4. The Space builds and starts `app.py` automatically. A request takes a few seconds on CPU (30 MC passes run as one batch;
   IG is the slow part).
5. Example images: if you add slices from the Figshare dataset as examples, credit it:
   Cheng J. et al., PLoS ONE 10(10): e0140381 (2015); dataset figshare 10.6084/m9.figshare.1512427, CC BY 4.0.
6. Put the disclaimer in the Space README: research demo, not a medical device.

Checkpoint md5: `{CKPT_MD5}`; trust thresholds were derived on the test split with manifest md5 `{MANIFEST_MD5}`.
''', encoding="utf-8")
import ast
ast.parse(APP)
print("wrote app.py (parses), requirements.txt, DEPLOY.md, best_model.pt, trust_thresholds.json")

## 9. Optional: launch the app inside Kaggle

Leave `RUN_GRADIO = False` for Run All and the commit. Set it to `True` in an interactive session
to get a temporary public link (`share=True`). Stop the cell when you're done; the link dies with
the session.

In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    import subprocess as _sp
    _sp.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    sys.path.insert(0, str(WORK))
    import runpy
    app_ns = runpy.run_path(str(WORK / "app.py"), run_name="kaggle_app")   # not __main__, so it doesn't auto-launch
    app_ns["demo"].launch(share=True)
else:
    print("RUN_GRADIO is False -- skipping the live server (app.py is on disk).")

## 10. The README

Assembled from the summaries and shown rendered below. Every number is a lookup, and every claim
goes through `claim()`. The static text describes *methods*, never *results*.

In [ ]:
nfaith = G("nb5", "n_with_faithfulness") or {}
n_audit = G("nb3", "n_model_audit")
n_test = test_metrics.get("n_test_images")
per_class = test_metrics.get("per_class", {})
pc_rows = "\n".join(f"| {c} | {f3(per_class.get(c, {}).get('precision'))} | {f3(per_class.get(c, {}).get('recall'))} | "
                    f"{f3(per_class.get(c, {}).get('f1-score'))} | {int(per_class.get(c, {}).get('support', 0))} |" for c in CLASS_NAMES)
if not per_class:
    TODOS.append("nb1:per_class")


def audit_row(metric, better):
    return (f"| {metric} ({better}) | " + " | ".join(S("nb3", f"per_method|{m}|{metric}") for m in METHODS + ["random"]) +
            f" | {S('nb3', f'paired_differences|gradcam_minus_integrated_gradients|{metric}', signed=True)} |")


stab_rows = "\n".join(
    f"| {s} | {S('nb3', f'per_method|gradcam|spearman_{s}')} | {S('nb3', f'per_method|integrated_gradients|spearman_{s}')} | "
    f"{S('nb3', f'per_method|gradcam|ssim_{s}')} | {S('nb3', f'per_method|integrated_gradients|ssim_{s}')} |" for s in SPECS)

faith_vs_random = []
for m in METHODS:
    d_ = claim("nb3", f"paired_differences|{m}_minus_random|deletion_auc",
               "**worse** than random on deletion", "**better** than random on deletion", "not distinguishable from random on deletion")
    i_ = claim("nb3", f"paired_differences|{m}_minus_random|insertion_auc",
               "**better** than random on insertion", "**worse** than random on insertion", "not distinguishable from random on insertion")
    faith_vs_random.append(f"- **{LBL[m]}** is {d_} ({S('nb3', f'paired_differences|{m}_minus_random|deletion_auc', signed=True)}) "
                           f"and {i_} ({S('nb3', f'paired_differences|{m}_minus_random|insertion_auc', signed=True)}).")

gc_vs_ig = G("nb5", "gradcam_vs_ig|verdict") or {}
gc_vs_ig_rows = "\n".join(f"| {axis} | {w} |" for axis, w in gc_vs_ig.items()) or "| `TODO(nb5:gradcam_vs_ig|verdict)` | |"

joint_lines = []
for m in METHODS:
    path = f"correlations|{m}|mc_conf_explained~faith_score"
    rel = claim("nb5", path, "more confident predictions have **more** faithful explanations",
                "more confident predictions have **less** faithful explanations",
                "**no detectable relation** between confidence and faithfulness")
    ct = G("nb5", f"crosstab|median|{m}") or {}
    joint_lines.append(f"- **{LBL[m]}:** {rel} (Spearman {S('nb5', path)}). Danger cell (confident + unfaithful + **wrong**, "
                       f"median split): **{ct.get('danger_confident_unfaithful_wrong', 'TODO')}** of "
                       f"{ct.get('n_errors_total_in_audit', 'TODO')} audited errors.")
usable = G("nb5", "uncertainty_flags_usable")
usable_txt = ("uncertainty **does** flag: " + "; ".join(f"{LBL.get(u['method'], u['method'])} - {u['target']} via {u['score']}" for u in usable)
              if usable else "**no** uncertainty measure flags untrustworthy explanations (no AUROC CI lies above 0.5)") if usable is not None else "`TODO(nb5:uncertainty_flags_usable)`"

unc_err = claim("nb4", "error_detection_auroc|predictive_entropy", "predictive entropy **does** separate errors from correct predictions",
                "predictive entropy is **inversely** related to errors", "predictive entropy does **not** reliably separate errors", null=0.5)
mc_vs_det = claim("nb4", "error_detection_auroc|predictive_entropy_minus_det_baseline", "MC entropy **beats** plain softmax confidence",
                  "MC entropy is **worse** than plain softmax confidence", "MC entropy is **not distinguishable** from plain softmax confidence")

bt_ = TH.get("backtest", {})
chain = """| # | notebook | consumes | produces | accelerator |
|---|---|---|---|---|
| 0 | `00_setup_and_data` | Figshare download | `manifest.csv` (patient-level split), `data/mat/*.mat`, `src/data/parse_mat.py` | None |
| 1 | `01_baseline_classifier` | nb 0 | `best_model.pt`, `test_predictions.csv`, `test_metrics.json`, `src/` (+dataset, transforms, classifier) | GPU |
| 2 | `02_explanations` | nb 0, 1 | `test_explanations.npz` (Grad-CAM, IG, masks), `meta.json`, `src/explain/` | GPU |
| 3 | `03_audit` | nb 0, 1, 2 | `audit_results.csv`, `audit_summary.json`, `audit_config.json`, `src/audit/`, tests | GPU |
| 4 | `04_uncertainty` | nb 0, 1 | `uncertainty_results.csv`, `uncertainty_summary.json`, `mc_samples.npz`, `src/uncertainty/`, tests | GPU |
| 5 | `05_joint_analysis` | nb 0, 2, 3, 4 | `joint_results.csv`, `joint_summary.json`, `correlations.csv` | None |
| 6 | `06_demo_and_readme` | nb 0, 1, 2, 3, 4, 5 | `README.md`, `project_summary.png`, `app.py`, trust thresholds | None (GPU optional) |"""

fig_lines = "\n".join(f"![{p.stem}](results/final/figures/{p.name})" for p in sorted((FINAL / "figures").glob("*.png")))

README = f"""# Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

**Thesis.** A saliency heatmap that *looks* convincing is not evidence that it is *trustworthy*. This project trains a
ResNet18 brain-tumor classifier, explains it with Grad-CAM and Integrated Gradients, and then **measures** whether those
explanations are faithful to the model, stable under tiny label-preserving perturbations, and localized on the tumor.
It also tests whether the model's own (MC-dropout) uncertainty tells us when an explanation should not be trusted.
Every number below is read programmatically from the pipeline's summary files and reported with a **patient-level
bootstrap 95% CI** (resampling the {G('nb4', 'n_patients') or 'TODO'} test patients, not images). A difference is
called a difference only when its CI excludes the null.

![project summary](results/final/project_summary.png)

## Dataset and split
- Figshare brain tumor dataset (Cheng et al., 2015; CC BY 4.0): 3064 T1-weighted contrast-enhanced axial slices,
  233 patients, three classes (meningioma, glioma, pituitary), with expert tumor masks.
- **Patient-level** stratified 70/15/15 split: every slice of a patient lives in exactly one split. Neighbouring slices
  are near-duplicates, so an image-level split would leak patients into the test set and inflate both accuracy and the
  explanation audit. Leakage is checked and recorded (`leakage_check.json`); the manifest's MD5 travels with every
  downstream artifact.

## Baseline classifier (notebook 1)
ImageNet-pretrained ResNet18, `Dropout(p) -> Linear` head, Adam + cosine schedule, early stopping on validation loss.
Test set: {n_test if n_test is not None else '`TODO(nb1:n_test_images)`'} images.

| metric | value |
|---|---|
| accuracy | {f3(test_metrics.get('accuracy'))} |
| balanced accuracy | {f3(test_metrics.get('balanced_accuracy'))} |
| macro-F1 | {f3(test_metrics.get('macro_f1'))} |

| class | precision | recall | F1 | support |
|---|---|---|---|---|
{pc_rows}

## The explanation audit (notebooks 2-3)
Model-based metrics on {n_audit if n_audit is not None else '`TODO(nb3:n_model_audit)`'} test images; localization on all.
A per-image random heatmap is the control.

**Faithfulness** (deletion/insertion, {audit_cfg.get('audit', {}).get('deletion_insertion_steps', 'TODO')} steps):

| metric | Grad-CAM | IG | random | Grad-CAM − IG |
|---|---|---|---|---|
{audit_row('deletion_auc', 'lower = more faithful')}
{audit_row('insertion_auc', 'higher = more faithful')}

{chr(10).join(faith_vs_random)}

**Stability** (similarity after aligning the perturbed map back; higher = more stable):

| perturbation | Grad-CAM Spearman | IG Spearman | Grad-CAM SSIM | IG SSIM |
|---|---|---|---|---|
{stab_rows}

**Localization** against the expert tumor mask:

| metric | Grad-CAM | IG | random | Grad-CAM − IG |
|---|---|---|---|---|
{audit_row('energy_ratio', '1 = chance')}
{audit_row('pointing_hit', 'hit rate')}
{audit_row('iou', 'threshold ' + str(audit_cfg.get('audit', {}).get('localization_threshold', 'TODO')))}
{audit_row('dice', 'same threshold')}

## MC-dropout uncertainty (notebook 4)
{G('nb4', 'settings|n_forward_passes') or 'TODO'} stochastic passes with only the head's dropout active (**last-layer MC-dropout**).

| | deterministic | MC-averaged |
|---|---|---|
| accuracy | {S('nb4', 'accuracy|deterministic')} | {S('nb4', 'accuracy|mc')} |
| ECE (15 bins) | {S('nb4', 'calibration|deterministic_ece')} | {S('nb4', 'calibration|mc_ece')} |
| NLL | {S('nb4', 'calibration|deterministic_nll')} | {S('nb4', 'calibration|mc_nll')} |

- Calibration: {G('nb4', 'calibration|verdict') or '`TODO(nb4:calibration|verdict)`'}.
- Error detection (AUROC): predictive entropy {S('nb4', 'error_detection_auroc|predictive_entropy')}, mutual information
  {S('nb4', 'error_detection_auroc|mutual_information')}, softmax baseline {S('nb4', 'error_detection_auroc|1-det_confidence (baseline)')}.
  {unc_err}; {mc_vs_det}.
- Confident-and-wrong (MC confidence >= 0.9): {(G('nb4', 'confident_and_wrong') or {}).get('count', 'TODO')} of
  {(G('nb4', 'confident_and_wrong') or {}).get('of_errors', 'TODO')} errors.

## Joint finding: is a confident prediction's explanation trustworthy? (notebook 5)
Faithfulness here is `faith_score = (ins - ins_random) - (del - del_random)` per image, which removes the mechanical
link between confidence and raw deletion/insertion AUCs. Images with faithfulness metrics: {nfaith or 'TODO'}.

{chr(10).join(joint_lines)}
- Does uncertainty flag untrustworthy explanations? {usable_txt}.

**Grad-CAM vs Integrated Gradients** (winner only where the paired CI excludes 0):

| axis | verdict |
|---|---|
{gc_vs_ig_rows}

Per-image agreement on faithfulness between the two explainers: Spearman {S('nb5', 'gradcam_vs_ig|per_image_agreement_spearman|faith_score')}.

## Demo and trust flag (notebook 6)
`predict_and_explain(image)` returns the class, MC confidence, entropy, mutual information, Grad-CAM and IG overlays and a
trust flag. The flag fires on any of: MC confidence < {f3(TH['min_confidence'], 4)}, predictive entropy > {f3(TH['max_entropy'], 4)},
or a Grad-CAM map covering > {f3(TH['max_gradcam_area'], 3) if TH['max_gradcam_area'] is not None else 'n/a'} of the image above
{LOC_THR}. These are the 5th/95th percentiles of correct test predictions and of well-localized maps. Back-test on that
same test set (optimistic): flags {f3(bt_.get('error_recall', {}).get('value'))} [{', '.join(f3(c) for c in bt_.get('error_recall', {}).get('ci95_patient_bootstrap', [None, None]))}]
of errors; accuracy {f3(bt_.get('accuracy_flagged', {}).get('value'))} when flagged vs {f3(bt_.get('accuracy_unflagged', {}).get('value'))} when not.
The mask-free localization proxy predicts a pointing-game miss with AUROC {f3(bt_.get('area_proxy_auroc_pointing_miss', {}).get('value'))}
[{', '.join(f3(c) for c in bt_.get('area_proxy_auroc_pointing_miss', {}).get('ci95_patient_bootstrap', [None, None]))}].
Deployable as a Gradio app (`app.py`, see `DEPLOY.md`).

![demo](results/final/demo_examples.png)

## Notebook chain and lineage
{chain}

Lineage: every artifact carries checkpoint MD5 `{CKPT_MD5}` and manifest MD5 `{MANIFEST_MD5}`; each notebook refuses to
join or quote outputs whose MD5s disagree.

## Limitations
- **Last-layer MC-dropout**: one dropout layer before the head, so the backbone features are identical across passes. This
  is head uncertainty, not full-network epistemic uncertainty (ensembles / full dropout / TTA are stronger alternatives).
- **Single architecture** (ResNet18) and **one dataset** ({G('nb4', 'n_patients') or 'TODO'} test patients); CIs are wide and
  findings may not transfer.
- **Sparse-map SSIM caveat**: SSIM scores ~1 on shared empty background, which inflates Grad-CAM's (sparse) stability.
- **Deletion-baseline bias**: black-patch deletion is off-distribution, and IG's black baseline matches it, which slightly
  favours IG on deletion. Insertion from a blurred image is reported alongside.
- **Per-image normalisation** of heatmaps affects threshold-based localization (IoU/Dice), not the rank or energy metrics.
- **Correlation is not causation**: the joint analysis shows associations across images, not mechanisms.
- The demo's trust thresholds were read off the **same test set** they are evaluated on; there is no untouched hold-out.

## Reproduce on Kaggle
Create one Kaggle notebook per file in `notebooks/`, run them in order, and **Save Version -> Save & Run All** each one
before attaching its output to the next. Inputs and accelerator per notebook are in the chain table above. Keep exactly
one version of each upstream notebook attached; the lineage checks stop the run on any mismatch.

## Citation
Dataset: Cheng J, Huang W, Cao S, Yang R, Yang W, Yun Z, Wang Z, Feng Q. *Enhanced Performance of Brain Tumor
Classification via Tumor Region Augmentation and Partition.* PLoS ONE 10(10): e0140381 (2015).
Figshare: doi:10.6084/m9.figshare.1512427. License: CC BY 4.0.

## Additional figures
{fig_lines}

*Research project; not a medical device.*
"""

(WORK / "README.md").write_text(README, encoding="utf-8")
display(Markdown(README))
print(f"\nREADME.md written ({len(README.split())} words). Unresolved TODO fields: {len(set(TODOS))}")
for t in sorted(set(TODOS)):
    print("   TODO", t)

## 11. Final verdict and hand-off

The paragraph below is **generated** from the same CIs as the README. Each clause is one branch of a
`claim()`, so it can't say more than the evidence supports. Use it as the abstract's last sentence
group, reworded if you like, but not strengthened.

In [ ]:
parts = [f"On a patient-level test split ({n_test if n_test is not None else 'TODO'} images), the ResNet18 reached "
         f"accuracy {f3(test_metrics.get('accuracy'))} (balanced {f3(test_metrics.get('balanced_accuracy'))})."]
for m in METHODS:
    ins = claim("nb3", f"paired_differences|{m}_minus_random|insertion_auc", "beat", "fell below", "did not distinguishably beat")
    parts.append(f"{LBL[m]} {ins} a random heatmap on insertion faithfulness "
                 f"({S('nb3', f'paired_differences|{m}_minus_random|insertion_auc', signed=True)}).")
for m in METHODS:
    parts.append(f"For {LBL[m]}, " + claim("nb5", f"correlations|{m}|mc_conf_explained~faith_score",
                 "higher confidence went with more faithful explanations",
                 "higher confidence went with less faithful explanations",
                 "confidence was not a reliable proxy for explanation faithfulness") +
                 f" (Spearman {S('nb5', f'correlations|{m}|mc_conf_explained~faith_score')}).")
parts.append("MC-dropout uncertainty " + ("flagged some untrustworthy explanations." if usable else
             "did not flag untrustworthy explanations." if usable is not None else "- TODO."))
dangers = [((G('nb5', f'crosstab|median|{m}') or {}).get('danger_confident_unfaithful_wrong')) for m in METHODS]
parts.append(f"Confidently wrong predictions with unfaithful explanations: {dangers[0]} (Grad-CAM) and {dangers[1]} (IG). "
             "All uncertainty is last-layer MC-dropout, on one architecture and one dataset.")
VERDICT = " ".join(parts)
print(VERDICT)
(FINAL / "verdict.txt").write_text(VERDICT, encoding="utf-8")

In [ ]:
expected = [WORK / "README.md", FINAL / "project_summary.png", WORK / "app.py", WORK / "requirements.txt", WORK / "DEPLOY.md",
            WORK / "best_model.pt", WORK / "trust_thresholds.json", FINAL / "demo_examples.png", FINAL / "verdict.txt",
            SRC / "demo" / "inference.py"]
for p in expected:
    assert p.exists() and p.stat().st_size > 0, f"missing {p}"
    print(f"  ok  {str(p.relative_to(WORK)):40s} {p.stat().st_size / 1024:9.1f} KB")
print("\nunresolved README TODOs:", len(set(TODOS)))

## Copy back into the GitHub repo

From this notebook's committed **Output**:

- [ ] `README.md` → repo root, replacing the old one. Check the TODO count printed above is 0 first.
- [ ] `results/final/project_summary.png`, `results/final/demo_examples.png`, `results/final/figures/*.png` →
  `results/final/`, keeping the paths, since the README links to them.
- [ ] `src/` (the merged tree: data, models, utils, explain, uncertainty, demo) → repo `src/`. This is the code the
  final results were produced with. Also copy `src/audit/` and `src/joint/` from notebooks 3 and 5's outputs, and
  `tests/` from notebooks 3 and 4.
- [ ] `app.py`, `requirements.txt`, `DEPLOY.md`, `trust_thresholds.json` → repo root, or a `demo/` folder; adjust the
  paths in `app.py` if you move them.
- [ ] `best_model.pt` → **not** into git directly (~45 MB). Use Git LFS, a GitHub Release asset, or the Hugging Face Space.
- [ ] The seven `.ipynb` files → `notebooks/`. Download each *committed version* (with outputs), so the repo shows
  results without anyone re-running.
- [ ] Optional: the summary JSONs from notebooks 1, 3, 4 and 5 → `results/`, so every README number can be traced to a file.

**Save & Run All** this notebook (accelerator None is fine) so that `README.md`, `project_summary.png`, `app.py` and the
deploy bundle persist as Output.